In [1]:
data = open('converted_chat_formatted.txt', 'r').read()

In [2]:
data[:1000]

"<start_of_turn>user\nHi!<end_of_turn>\n<start_of_turn>model\nWhat is your favorite holiday?<end_of_turn>\n<start_of_turn>user\none where I get to meet lots of different people.<end_of_turn>\n<start_of_turn>model\nWhat was the most number of people you have ever met during a holiday?<end_of_turn>\n<start_of_turn>user\nHard to keep a count. Maybe 25.<end_of_turn>\n<start_of_turn>model\nWhich holiday was that?<end_of_turn>\n<start_of_turn>user\nI think it was Australia<end_of_turn>\n<start_of_turn>model\nDo you still talk to the people you met?<end_of_turn>\n<start_of_turn>user\nNot really. The interactions are usually short-lived but it's fascinating to learn where people are coming from and what matters to them<end_of_turn>\n<start_of_turn>model\nYea, me too. I feel like God often puts strangers in front of you, and gives you an opportunity to connect with them in that moment in deeply meaningful ways. Do you ever feel like you know things about strangers without them telling you?<end_

In [3]:
vocabs = list(sorted(set(data)))

In [4]:
stoi = {s:i for i,s in enumerate(vocabs)}
itos = {i:s for i,s in enumerate(vocabs)}

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [6]:
# model hyperParameters
n_embed    = 180
n_hidden   = 180
num_heads  = 4
vocab_size = len(vocabs)
batch_size = 16
block_size = 300
lr         = 1e-3


In [7]:
# import tools 
from tool import LearnedPE as lpe
from tool import ResidualBlock as rb
from tool import generate as gn
from tool import forward as fw
from tool import get_data as gd
from tool import fit
from tool import capture
from tool import encode
from tool import show_chat

In [8]:
class miniLM(nn.Module):
  """mini language model architecture"""
  forward  = fw
  generate = gn
  get_data = gd
  fit      = fit
  enctrain = encode(data[:int(len(data)*0.9)], stoi)
  encval   = encode(data[int(len(data)*0.9):], stoi)
  show_chat= show_chat

  def __init__(self):

    super().__init__()
    self.chat = []
    self.embed= nn.Sequential(
      nn.Embedding(vocab_size, n_embed), lpe(300, n_embed),                               # 1st
    )
    self.rb1  = rb(cap=capture, just_capture=True)                                        # 2nd
    self.mha1 = nn.MultiheadAttention(n_embed,  num_heads, dropout=0.2, batch_first=True) # 3rd
    self.ffn1 = nn.Sequential(nn.Linear(n_embed, n_hidden, bias=False),                   # 4th
                                   nn.LayerNorm(n_hidden), nn.Tanh())
    self.rb2  = rb(cap=capture)                                                           # 5th
    self.mha2 = nn.MultiheadAttention(n_hidden, num_heads, dropout=0.2, batch_first=True)# 6th
    self.ffn2 = nn.Sequential(nn.Linear(n_hidden,n_hidden, bias=False),                  # 7th
                                  nn.LayerNorm(n_hidden), nn.ReLU())
    self.rb3  = rb(cap=capture)                                                           # 8th
    self.lgts = nn.Linear(n_hidden, vocab_size, bias=True)                                # 9th

    # load the saved model
    #self.load_state_dict(torch.load("miniLM.pt", weights_only=True))
    self.eval()
    print(f"total parameters = {sum(p.nelement() for p in self.parameters())}")

    self.block_size= block_size

In [9]:
model = miniLM()

total parameters = 418065


In [14]:
model.fit(2000, batch_size) # ~50k epochs when batch_size == 1

epoch:100   | loss=1.1000   |  val loss=1.0970
epoch:200   | loss=1.0493   |  val loss=1.0492
epoch:300   | loss=1.0733   |  val loss=0.9796
epoch:400   | loss=1.0650   |  val loss=1.0120
epoch:500   | loss=1.1091   |  val loss=1.0280
epoch:600   | loss=1.1041   |  val loss=1.0065
epoch:700   | loss=1.0869   |  val loss=0.9246
epoch:800   | loss=1.1289   |  val loss=0.9008
epoch:900   | loss=1.1093   |  val loss=1.0520
epoch:1000   | loss=1.1585   |  val loss=0.9985
epoch:1100   | loss=1.0741   |  val loss=1.0255
epoch:1200   | loss=0.9695   |  val loss=1.0425
epoch:1300   | loss=1.0611   |  val loss=1.0183
epoch:1400   | loss=0.9813   |  val loss=1.1010
epoch:1500   | loss=1.1468   |  val loss=1.0396
epoch:1600   | loss=1.0274   |  val loss=1.0072
epoch:1700   | loss=1.1032   |  val loss=0.9579
epoch:1800   | loss=1.0946   |  val loss=1.0711
epoch:1900   | loss=1.1539   |  val loss=1.0947
epoch:2000   | loss=1.0464   |  val loss=1.0122


In [16]:
#model.generate(stoi, itos, block_size, use_memory=True)